## Cell 1 — Imports & Logging
Loads all required libraries. `geopandas` handles shapefile reading
and spatial operations. `requests` handles HTTP downloads.
Run the pip install first if either library is missing.

In [0]:
%pip install geopandas requests --quiet

import io
import os
import json
import logging
import zipfile
import tempfile
import requests

from datetime import datetime, timezone
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s"
)
logger = logging.getLogger("stl_geo_transform")

print("✓ Imports loaded")

## Cell 2 — Configuration
All paths and source URLs in one place. The output directory
is where both GeoJSON files will be written. Update
`GEO_OUTPUT_DIR` if your Volume path differs.

In [0]:
# ── Output directory ──────────────────────────────────────────
# Both GeoJSON files land here. Downstream transform notebooks
# read from this path.
GEO_OUTPUT_DIR = "/Volumes/workspace/default/raw/geo"

# Derived file paths
NEIGHBORHOODS_PATH = f"{GEO_OUTPUT_DIR}/stl_neighborhoods.geojson"
OVERLAY_PATH       = f"{GEO_OUTPUT_DIR}/tract_neighborhood_overlay.geojson"

# ── Source URLs ───────────────────────────────────────────────
# Neighborhoods shapefile — tried in order, first success wins
NEIGHBORHOOD_URLS = [
    "https://static.stlouis-mo.gov/open-data/planning/neighborhoods/neighborhoods.zip",
    "https://dynamic.stlouis-mo.gov/opendata/downloads/nbrhds_wards.zip",
]

# Census TIGER tract shapefile for Missouri (state FIPS 29)
TIGER_TRACT_URL = (
    "https://www2.census.gov/geo/tiger/TIGER2025/TRACT/tl_2025_29_tract.zip"
)

# St. Louis jurisdiction FIPS codes
STL_CITY_FIPS   = "510"
STL_COUNTY_FIPS = "189"

# Browser headers to avoid 403 blocks on government portals
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
}

os.makedirs(GEO_OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory: {GEO_OUTPUT_DIR}")

## Cell 3 — Neighborhood Lookup Table
Authoritative mapping of NHD_NUM numeric codes to neighborhood names.
Sourced directly from the City of St. Louis neighborhoods shapefile
(NHD_NAME field). Verified against the downloaded shapefile on
2026-04-19.

These codes appear as the `neighborhood` field in SLMPD crime data
and as `NBRHD` in the parcel DBF.

Note: The shapefile contains 79 residential neighborhoods after
filtering 9 parks/cemeteries. Parks are NOT included here since
they don't appear in crime or parcel data meaningfully.

In [0]:
# Parks and cemeteries present in the raw shapefile (88 total features)
# but excluded from residential analyses
NON_RESIDENTIAL = {
    "Bellefontaine/Calvary Cemetery",
    "Carondelet Park",
    "Fairground Park",
    "Forest Park",
    "Missouri Botanical Garden",
    "O'Fallon Park",
    "Penrose Park",
    "Tower Grove Park",
    "Willmore Park",
}

# Authoritative NHD_NUM → NHD_NAME mapping
# Source: City of St. Louis neighborhoods shapefile, verified 2026-04-19
# Code 0 = ungeocoded / missing (appears in older SLMPD records)
NEIGHBORHOOD_LOOKUP: dict[int, str] = {
    0:  "Unknown",
    1:  "Carondelet",
    2:  "Patch",
    3:  "Holly Hills",
    4:  "Boulevard Heights",
    5:  "Bevo Mill",
    6:  "Princeton Heights",
    7:  "Southampton",
    8:  "St. Louis Hills",
    9:  "Lindenwood Park",
    10: "Ellendale",
    11: "Clifton Heights",
    12: "The Hill",
    13: "Southwest Garden",
    14: "North Hampton",
    15: "Tower Grove South",
    16: "Dutchtown",
    17: "Mount Pleasant",
    18: "Marine Villa",
    19: "Gravois Park",
    20: "Kosciusko",
    21: "Soulard",
    22: "Benton Park",
    23: "McKinley Heights",
    24: "Fox Park",
    25: "Tower Grove East",
    26: "Compton Heights",
    27: "Shaw",
    28: "Botanical Heights",
    29: "Tiffany",
    30: "Benton Park West",
    31: "The Gate District",
    32: "Lafayette Square",
    33: "Peabody Darst Webbe",
    34: "LaSalle Park",
    35: "Downtown",
    36: "Downtown West",
    37: "Midtown",
    38: "Central West End",
    39: "Forest Park South East",
    40: "Kings Oak",
    41: "Cheltenham",
    42: "Clayton-Tamm",
    43: "Franz Park",
    44: "Hi-Pointe",
    45: "Wydown Skinker",
    46: "Skinker DeBaliviere",
    47: "DeBaliviere Place",
    48: "West End",
    49: "Visitation Park",
    50: "Wells Goodfellow",
    51: "Academy",
    52: "Kingsway West",
    53: "Fountain Park",
    54: "Lewis Place",
    55: "Kingsway East",
    56: "Greater Ville",
    57: "The Ville",
    58: "Vandeventer",
    59: "Jeff Vanderlou",
    60: "St. Louis Place",
    61: "Carr Square",
    62: "Columbus Square",
    63: "Old North St. Louis",
    64: "Near North Riverfront",
    65: "Hyde Park",
    66: "College Hill",
    67: "Fairground Neighborhood",
    68: "O'Fallon",
    69: "Penrose",
    70: "Mark Twain I-70 Industrial",
    71: "Mark Twain",
    72: "Walnut Park East",
    73: "North Pointe",
    74: "Baden",
    75: "Riverview",
    76: "Walnut Park West",
    77: "Covenant Blu-Grand Center",
    78: "Hamilton Heights",
    79: "North Riverfront",
}

# Residential-only subset — used in crime rate and vacancy calculations
RESIDENTIAL_NEIGHBORHOODS = {
    code: name
    for code, name in NEIGHBORHOOD_LOOKUP.items()
    if name not in NON_RESIDENTIAL and code != 0
}

print(f"✓ Lookup table loaded: {len(NEIGHBORHOOD_LOOKUP)} total codes")
print(f"✓ Residential neighborhoods: {len(RESIDENTIAL_NEIGHBORHOODS)}")
print(f"\nSpot checks:")
print(f"  Code 35 → {NEIGHBORHOOD_LOOKUP[35]}")   # should be Downtown
print(f"  Code 38 → {NEIGHBORHOOD_LOOKUP[38]}")   # should be Central West End
print(f"  Code 21 → {NEIGHBORHOOD_LOOKUP[21]}")   # should be Soulard
print(f"  Code 16 → {NEIGHBORHOOD_LOOKUP[16]}")   # should be Dutchtown

## Cell 4 — Download Helper
Reusable function for downloading ZIP files with retry logic.
Used for both the neighborhoods shapefile and the Census TIGER
tract file. Returns raw bytes or None on failure — never raises.

In [0]:
def download_zip(url: str, timeout: int = 120) -> bytes | None:
    """
    Download a ZIP file from the given URL with up to 3 retries.
    Returns raw bytes on success, None if all attempts fail.

    Args:
        url:     Full URL to the ZIP file.
        timeout: Per-request timeout in seconds. Set higher for
                 large files like the TIGER tract shapefile (~50MB).
    """
    for attempt in range(1, 4):
        try:
            logger.info(f"  GET {url} (attempt {attempt}/3)")
            resp = requests.get(
                url,
                headers=BROWSER_HEADERS,
                timeout=timeout
            )
            if resp.status_code == 200:
                logger.info(f"  ✓ {len(resp.content) / 1e6:.2f} MB downloaded")
                return resp.content
            if resp.status_code in (403, 404):
                logger.warning(f"  HTTP {resp.status_code} — not retrying")
                return None
            logger.warning(f"  HTTP {resp.status_code} on attempt {attempt}/3")
        except requests.Timeout:
            logger.warning(f"  Timeout on attempt {attempt}/3")
        except requests.RequestException as e:
            logger.warning(f"  Request error: {e}")
    logger.error(f"  All attempts failed for {url}")
    return None


def extract_shp(zip_bytes: bytes, tmpdir: str) -> str | None:
    """
    Extract a ZIP to tmpdir and return the path to the first .shp file.
    Returns None if no shapefile found or ZIP is invalid.
    """
    try:
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            logger.info(f"  ZIP contents: {zf.namelist()}")
            zf.extractall(tmpdir)
    except zipfile.BadZipFile as e:
        logger.error(f"  Bad ZIP: {e}")
        return None

    shp_files = list(Path(tmpdir).rglob("*.shp"))
    if not shp_files:
        logger.error("  No .shp file found in ZIP")
        return None

    return str(shp_files[0])


print("✓ download_zip() and extract_shp() defined")

## Cell 5 — Fallback GeoJSON Writer
If the neighborhoods shapefile download fails, this writes a
geometry-free GeoJSON from the hardcoded lookup table. Downstream
name mapping still works, but spatial operations and map rendering
will not be available. Think of it as a graceful degradation.

In [0]:
def write_fallback_geojson(output_path: str) -> None:
    """
    Write a name-only GeoJSON (no geometry) from the hardcoded lookup.

    Used when the shapefile download fails. Downstream joins on
    neighborhood_id still work, but no geometry is available for
    spatial operations or map rendering.

    Args:
        output_path: Path to write the fallback GeoJSON file.
    """
    features = [
        {
            "type": "Feature",
            "geometry": None,
            "properties": {
                "neighborhood_id":   code,
                "neighborhood_name": name,
            }
        }
        for code, name in RESIDENTIAL_NEIGHBORHOODS.items()
    ]

    gj = {"type": "FeatureCollection", "features": features}

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(gj, f, indent=2)

    logger.warning(
        f"⚠ Wrote geometry-free fallback GeoJSON → {output_path}\n"
        f"  Name mapping works but spatial operations unavailable."
    )


print("✓ write_fallback_geojson() defined")

## Cell 6 — Build Neighborhoods GeoJSON
Downloads the neighborhoods shapefile, filters out 9 parks and
cemeteries, reprojects to WGS84, standardizes column names, and
saves as `stl_neighborhoods.geojson`.

The output has two key columns:
- `neighborhood_id`   — numeric code (matches crime/parcel data)
- `neighborhood_name` — official name (e.g. "Soulard")

Falls back to geometry-free GeoJSON if all download URLs fail.

In [0]:
import geopandas as gpd

def build_neighborhoods_geojson(output_path: str) -> bool:
    """
    Download, clean, and save the STL neighborhoods shapefile as GeoJSON.

    Steps:
      1. Try each URL in NEIGHBORHOOD_URLS until one succeeds
      2. Extract shapefile from ZIP
      3. Detect the name and numeric ID columns
      4. Filter out parks and cemeteries
      5. Reproject to WGS84 (EPSG:4326)
      6. Standardize column names
      7. Save as GeoJSON

    Returns:
        True if saved with geometry, False if fallback was used.
    """
    # ── Step 1: Download the shapefile ZIP ───────────────────
    zip_bytes = None
    for url in NEIGHBORHOOD_URLS:
        logger.info(f"Trying: {url}")
        zip_bytes = download_zip(url)
        if zip_bytes is not None:
            break

    if zip_bytes is None:
        logger.warning("All download URLs failed — using fallback")
        write_fallback_geojson(output_path)
        return False

    with tempfile.TemporaryDirectory() as tmpdir:

        # ── Step 2: Extract shapefile ─────────────────────────
        shp_path = extract_shp(zip_bytes, tmpdir)
        if shp_path is None:
            write_fallback_geojson(output_path)
            return False

        # ── Step 3: Read and inspect columns ─────────────────
        gdf = gpd.read_file(shp_path)
        logger.info(f"  Loaded {len(gdf)} features, CRS: {gdf.crs}")
        logger.info(f"  Columns: {list(gdf.columns)}")

        # Detect the name column — try common variants
        name_col = next(
            (c for c in ["NHD_NAME", "NAME", "NEIGHBORHD", "LABEL", "NHOOD"]
             if c in gdf.columns),
            None
        )
        if name_col is None:
            logger.error(f"Cannot find name column in {list(gdf.columns)}")
            write_fallback_geojson(output_path)
            return False

        # Detect numeric ID column — try common variants
        num_col = next(
            (c for c in ["NHD_NUM", "NBRHD", "ID", "NUM", "OBJECTID"]
             if c in gdf.columns),
            None
        )

        logger.info(f"  Name column: {name_col}, ID column: {num_col or 'none found'}")

        # ── Step 4: Filter out non-residential ───────────────
        gdf = gdf[~gdf[name_col].isin(NON_RESIDENTIAL)].copy()
        logger.info(f"  After filtering parks: {len(gdf)} residential neighborhoods")

        # ── Step 5: Reproject to WGS84 ───────────────────────
        if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
            logger.info("  Reprojected to EPSG:4326")

        # ── Step 6: Standardize column names ─────────────────
        rename = {name_col: "neighborhood_name"}
        if num_col:
            rename[num_col] = "neighborhood_id"
        gdf = gdf.rename(columns=rename)

        # If no numeric ID in shapefile, derive from lookup table
        if "neighborhood_id" not in gdf.columns:
            name_to_id = {v: k for k, v in NEIGHBORHOOD_LOOKUP.items()}
            gdf["neighborhood_id"] = gdf["neighborhood_name"].map(name_to_id)
            logger.info("  Derived neighborhood_id from lookup table")

        # Keep only essential columns
        keep = ["neighborhood_id", "neighborhood_name", "geometry"]
        gdf  = gdf[[c for c in keep if c in gdf.columns]]

        # ── Step 7: Save ──────────────────────────────────────
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        gdf.to_file(output_path, driver="GeoJSON")
        logger.info(f"  ✓ Saved {len(gdf)} neighborhoods → {output_path}")

    return True


print("✓ build_neighborhoods_geojson() defined")

## Cell 7 — Run Neighborhoods Build

Executes the neighborhoods GeoJSON build and reports whether
real geometry was produced or the fallback was used.
Check the log output to see which URL succeeded.

In [0]:
print("Building neighborhoods GeoJSON...")
print("=" * 60)

neighborhoods_success = build_neighborhoods_geojson(NEIGHBORHOODS_PATH)

if neighborhoods_success:
    # Read back and display a sample to confirm it looks right
    nhoods = gpd.read_file(NEIGHBORHOODS_PATH)
    print(f"\n✓ Success — {len(nhoods)} neighborhoods with geometry")
    print(f"  Columns: {list(nhoods.columns)}")
    display(nhoods[["neighborhood_id", "neighborhood_name"]].sort_values("neighborhood_id"))
else:
    print("\n⚠ Fallback used — geometry-free GeoJSON written")
    print("  Name mapping will work but spatial operations unavailable")
    print("  Check if the shapefile URL is accessible from this cluster")

## Cell 8 — Build Tract-Neighborhood Overlay

Downloads Census TIGER tracts for Missouri, filters to St. Louis
City and County, then computes the area-weighted spatial intersection
with neighborhood boundaries.

The result has one row per tract-neighborhood pair, with an
`area_weight` column representing what fraction of the tract's area
falls in that neighborhood. This weight is used downstream to
apportion Census population from tract level to neighborhood level.

**Example:** If 60% of a census tract falls in Soulard and 40% in
Benton Park, then 60% of that tract's population is attributed to
Soulard in the crime rate calculation.

Note: This cell requires that Cell 7 produced a GeoJSON with real
geometry. It will be skipped if the fallback was used.

In [0]:
def build_tract_neighborhood_overlay(
    neighborhoods_path: str,
    output_path: str,
) -> bool:
    """
    Build an area-weighted overlay of Census tracts and neighborhoods.

    Steps:
      1. Load neighborhoods GeoJSON (must have real geometry)
      2. Download Census TIGER Missouri tract shapefile
      3. Filter to STL City + County tracts only
      4. Reproject both layers to EPSG:26915 (UTM Zone 15N)
         — projected CRS required for accurate area calculations
      5. Compute tract areas before intersection
      6. Spatial intersection (one row per tract-neighborhood pair)
      7. Compute area_weight = intersect_area / tract_area
      8. Reproject back to WGS84 and save as GeoJSON

    Returns:
        True on success, False on failure.
    """
    # ── Step 1: Load neighborhoods ────────────────────────────
    if not os.path.exists(neighborhoods_path):
        logger.error(f"Neighborhoods file not found: {neighborhoods_path}")
        return False

    neighborhoods = gpd.read_file(neighborhoods_path)

    # Verify we have real geometry (not the fallback version)
    if neighborhoods.geometry.isna().all():
        logger.error(
            "Neighborhoods GeoJSON has no geometry. "
            "Cannot build overlay without real boundaries."
        )
        return False

    logger.info(f"  Loaded {len(neighborhoods)} neighborhoods")

    # ── Step 2: Download Census TIGER tracts ─────────────────
    logger.info(f"Downloading Census TIGER tracts (~50MB, may take a moment)...")
    zip_bytes = download_zip(TIGER_TRACT_URL, timeout=180)
    if zip_bytes is None:
        logger.error("Census TIGER tract download failed")
        return False

    with tempfile.TemporaryDirectory() as tmpdir:
        shp_path = extract_shp(zip_bytes, tmpdir)
        if shp_path is None:
            return False

        # ── Step 3: Filter to STL City + County ──────────────
        tracts = gpd.read_file(shp_path)
        logger.info(f"  Loaded {len(tracts)} Missouri tracts")

        tracts = tracts[
            tracts["COUNTYFP"].isin([STL_CITY_FIPS, STL_COUNTY_FIPS])
        ].copy()
        logger.info(f"  After filtering to STL: {len(tracts)} tracts")

        # ── Step 4: Reproject to projected CRS ───────────────
        # EPSG:26915 = UTM Zone 15N — covers the St. Louis region
        # Area calculations in geographic CRS (degrees) are inaccurate
        projected_crs      = "EPSG:26915"
        tracts_proj        = tracts.to_crs(projected_crs)
        neighborhoods_proj = neighborhoods.to_crs(projected_crs)

        # ── Step 5: Compute tract areas ───────────────────────
        tracts_proj["tract_area_m2"] = tracts_proj.geometry.area

        # ── Step 6: Spatial intersection ──────────────────────
        logger.info("  Computing spatial intersection (this may take ~30 seconds)...")
        overlay = gpd.overlay(
            tracts_proj,
            neighborhoods_proj,
            how="intersection"
        )
        logger.info(f"  Intersection: {len(overlay)} tract-neighborhood pairs")

        # ── Step 7: Compute area weights ──────────────────────
        overlay["intersect_area_m2"] = overlay.geometry.area
        overlay["area_weight"] = (
            overlay["intersect_area_m2"] / overlay["tract_area_m2"]
        )

        # ── Step 8: Reproject to WGS84 and save ──────────────
        overlay = overlay.to_crs(epsg=4326)

        keep_cols = [
            "GEOID",               # 11-digit Census tract FIPS
            "COUNTYFP",            # City (510) vs County (189)
            "neighborhood_id",
            "neighborhood_name",
            "tract_area_m2",
            "intersect_area_m2",
            "area_weight",
            "geometry",
        ]
        overlay = overlay[[c for c in keep_cols if c in overlay.columns]]

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        overlay.to_file(output_path, driver="GeoJSON")
        logger.info(f"  ✓ Saved overlay → {output_path}")

    return True


print("✓ build_tract_neighborhood_overlay() defined")

## Cell 9 — Run Tract-Neighborhood Overlay

Executes the tract-neighborhood overlay build. Only runs if
Cell 7 produced a GeoJSON with real geometry. Prints a sample
of the area weights to confirm the overlay looks correct —
weights for a given tract should sum to approximately 1.0.

In [0]:
if not neighborhoods_success:
    print("⚠ Skipping overlay — neighborhoods file has no geometry")
    print("  Resolve the shapefile download issue and re-run Cell 7 first")
    overlay_success = False
else:
    print("Building tract-neighborhood overlay...")
    print("=" * 60)

    overlay_success = build_tract_neighborhood_overlay(
        NEIGHBORHOODS_PATH,
        OVERLAY_PATH,
    )

    if overlay_success:
        overlay = gpd.read_file(OVERLAY_PATH)
        print(f"\n✓ Overlay complete — {len(overlay):,} tract-neighborhood pairs")
        print(f"\nSample rows:")
        display(
            overlay[["GEOID", "neighborhood_name", "area_weight"]]
            .sort_values("GEOID")
            .head(10)
        )

        # Verify weights sum to ~1.0 per tract
        weight_check = (
            overlay.groupby("GEOID")["area_weight"]
            .sum()
            .describe()
        )
        print(f"\nArea weight sum per tract (should be ~1.0):")
        print(weight_check)

## Cell 10 — Lookup Table Validation
Verifies the hardcoded neighborhood lookup table is internally
consistent and matches what we'd expect from the city data.
Useful as a quick sanity check after any edits to Cell 3.

In [0]:
print("=== LOOKUP TABLE VALIDATION ===\n")

# Count residential vs non-residential
residential_count    = len(RESIDENTIAL_NEIGHBORHOODS)
non_residential_count = len([
    v for v in NEIGHBORHOOD_LOOKUP.values()
    if v in NON_RESIDENTIAL
])

print(f"Total codes (0–88):       {len(NEIGHBORHOOD_LOOKUP)}")
print(f"Residential neighborhoods: {residential_count}")
print(f"Parks/cemeteries:          {non_residential_count}")
print(f"Code 0 (Unknown):          1")
print(f"Expected total:            {residential_count + non_residential_count + 1}")

# Verify no duplicate names in residential neighborhoods
names = list(RESIDENTIAL_NEIGHBORHOODS.values())
dupes = [n for n in names if names.count(n) > 1]
if dupes:
    print(f"\n⚠ Duplicate names found: {set(dupes)}")
else:
    print(f"\n✓ No duplicate names")

# Show the full residential lookup as a table
import pandas as pd
lookup_df = pd.DataFrame(
    [(k, v) for k, v in sorted(RESIDENTIAL_NEIGHBORHOODS.items())],
    columns=["neighborhood_id", "neighborhood_name"]
)
print(f"\nFull residential neighborhood lookup ({len(lookup_df)} entries):")
display(lookup_df)

## Cell 11 — Summary
Reports what was built and what paths downstream notebooks
should use when reading the geo reference data.

In [0]:
print("=" * 60)
print("GEO TRANSFORM SUMMARY")
print("=" * 60)

print(f"\nNeighborhoods GeoJSON:")
print(f"  Path:    {NEIGHBORHOODS_PATH}")
print(f"  Status:  {'✓ With geometry' if neighborhoods_success else '⚠ Fallback (no geometry)'}")

print(f"\nTract-Neighborhood Overlay:")
print(f"  Path:    {OVERLAY_PATH}")
print(f"  Status:  {'✓ Built' if overlay_success else '⚠ Not built'}")

print(f"\nLookup Table:")
print(f"  Residential neighborhoods: {len(RESIDENTIAL_NEIGHBORHOODS)}")
print(f"  Use: NEIGHBORHOOD_LOOKUP[int(neighborhood_code)]")

print(f"\nDownstream notebooks should read from:")
print(f"  {NEIGHBORHOODS_PATH}")
print(f"  {OVERLAY_PATH}")
print(f"\nOr import the lookup directly:")
print(f"  from transform.geo import NEIGHBORHOOD_LOOKUP, RESIDENTIAL_NEIGHBORHOODS")